# ray-parametric-form — ex3: reflect rays off a ground plane and trace the bounce

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `ray-parametric-form`. Running the final beacon cell reports progress against the `Geometry: Ray parametric form` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Geometry: Ray parametric form` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`ray-parametric-form`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "ray-parametric-form"
DD_SUBTOPIC = "Geometry: Ray parametric form"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Ray parametric form — quick refresher

A ray is `R(u) = O + u * D` where `O` is the origin and `D` is the direction. For `u >= 0` the equation traces the ray forward; `u < 0` is behind the origin. To **reflect** a ray off a plane with normal `n`, you keep the origin at the hit point and replace the direction with `D - 2 * (D · n) * n` (the component along `n` flips sign).

### Exercise 3 — reflect rays off a ground plane and trace the bounce

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Create
> LO: Compose the parametric ray equation with a planar-reflection update to trace a batch of rays through one bounce off the y=0 plane, returning both the hit-point batch and the post-bounce ray batch.
> Keywords: ray, reflection, two-segment, ground-plane, integrative
> ```

**KCs targeted:** `ray-eval-broadcast-batch`, `ray-origin-direction-storage`

Existing ex1 / ex2 evaluate a single ray equation. This one *composes* the equation with a reflection update — a two-stage pipeline.

Implement `ex3_bounce_off_ground(rays)`:

1. `rays` has shape `(B, 2, 3)` — `rays[:, 0]` are origins `O`, `rays[:, 1]` are directions `D`. All `D` have `D.y < 0` (pointing down). All `O` have `O.y > 0` (above the ground).
2. Solve for the hit parameter `u_hit = -O.y / D.y` (one scalar per ray).
3. Hit point `H = O + u_hit * D` via the parametric form — every `H.y` must be ~0.
4. Reflect `D` across the ground-plane normal `n = (0,1,0)`: `D_reflected = D - 2 * (D · n) * n`. Because `n=(0,1,0)`, this simply flips the y component.
5. Return a new `(B, 2, 3)` `rays_after` tensor whose origin row is `H` and direction row is `D_reflected`.

Output dtype must be `float32`. After the bounce, every `D_reflected.y` must be positive (heading up).

In [ ]:
def ex3_bounce_off_ground(rays: Tensor) -> Tensor:
    """Return (B,2,3) post-bounce rays after reflecting off y=0."""
    raise NotImplementedError()


def _test_ex3():
    # Two hand-checked rays.
    rays = t.tensor([
        # ray 0 — origin (0, 2, 0), straight down → hits (0,0,0), reflects to (0,1,0)
        [[0.0, 2.0, 0.0], [0.0, -1.0, 0.0]],
        # ray 1 — origin (0, 3, 0), direction (1, -1, 0) → hits (3,0,0), reflects to (1,1,0)
        [[0.0, 3.0, 0.0], [1.0, -1.0, 0.0]],
    ])
    out = ex3_bounce_off_ground(rays)
    assert out.shape == (2, 2, 3), f'expected (2,2,3), got {tuple(out.shape)}'
    assert out.dtype == t.float32, f'expected float32, got {out.dtype}'
    expected = t.tensor([
        [[0.0, 0.0, 0.0], [0.0, 1.0, 0.0]],
        [[3.0, 0.0, 0.0], [1.0, 1.0, 0.0]],
    ])
    assert t.allclose(out, expected, atol=1e-5), f'value mismatch:\n{out}\nvs\n{expected}'
    # Every hit must be on the ground.
    assert t.allclose(out[:, 0, 1], t.zeros(2), atol=1e-5), 'hit-point y must be 0'
    # Every reflected direction must be heading up.
    assert (out[:, 1, 1] > 0).all(), 'reflected direction must have positive y'

    # Batch smoke test on a fan of rays + bounce visualization.
    rng = t.Generator().manual_seed(7)
    B = 40
    origins = t.stack([t.linspace(-3, 3, B), t.full((B,), 2.5), t.zeros(B)], dim=1)
    directions = t.stack([
        0.4 * t.randn(B, generator=rng),
        t.full((B,), -1.0),
        t.zeros(B),
    ], dim=1)
    big_rays = t.stack([origins, directions], dim=1)
    big_out = ex3_bounce_off_ground(big_rays)
    hits = big_out[:, 0]
    reflected = big_out[:, 1]
    assert t.allclose(hits[:, 1], t.zeros(B), atol=1e-5)
    assert (reflected[:, 1] > 0).all()

    # --- Bounce visualization (X-Y projection) ---
    fig, ax = plt.subplots(figsize=(6, 4))
    for i in range(B):
        O = origins[i].numpy(); H = hits[i].numpy()
        R = (hits[i] + 1.5 * reflected[i]).numpy()  # trace 1.5 units after bounce
        ax.plot([O[0], H[0]], [O[1], H[1]], color='tab:blue', alpha=0.5, linewidth=0.7)
        ax.plot([H[0], R[0]], [H[1], R[1]], color='tab:orange', alpha=0.5, linewidth=0.7)
    ax.axhline(0, color='k', linewidth=1)
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.set_title(f'ex3 ray bounce off y=0  (blue=incoming, orange=reflected, B={B})')
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    _dd_passed.add('ex3')
    print("ex3 ✓")

_test_ex3()

<details><summary>Solution</summary>

```python
def ex3_bounce_off_ground(rays: Tensor) -> Tensor:
    O = rays[:, 0]
    D = rays[:, 1]
    u_hit = -O[:, 1] / D[:, 1]                     # (B,)
    H = O + u_hit.unsqueeze(-1) * D                # (B, 3)
    n = t.tensor([0.0, 1.0, 0.0])                  # ground normal
    D_reflected = D - 2 * (D @ n).unsqueeze(-1) * n
    return t.stack([H, D_reflected], dim=1).to(t.float32)
```

**Two stages of the parametric form.** First we *consume* the ray equation to find the hit point (`H = O + u_hit * D`); then we *re-emit* a new ray rooted at the hit point with a transformed direction. The atom is the equation; the drill is the composition.

**Why the formula `D - 2 (D·n) n` works.** Decompose `D` into `D_parallel + D_perp` where `D_parallel = (D·n) n` (along the normal) and `D_perp` is in the plane. Reflection negates the parallel component → `D - 2 (D·n) n`. For our axis-aligned normal `(0,1,0)`, this collapses to flipping the y component, but the general form is what real raytracers use for arbitrary plane normals.

**`u_hit` shape gotcha.** `O[:,1]` and `D[:,1]` are both `(B,)`, so `u_hit` is `(B,)`. To multiply against `D` of shape `(B,3)` you need an `unsqueeze(-1)` — exactly the same broadcast pattern ex1 and ex2 used.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()